In [35]:
import sys
import pandas as pd
from pathlib import Path

# This moves from notebooks/ up one level to project root
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

print("Using project root:", PROJECT_ROOT)


Using project root: C:\Users\nathan.taylor\Jupyter Notebooks\outlier_pipeline


In [36]:
from src.runner import run_all_jobs

CONFIG_PATH = PROJECT_ROOT / "configs" / "jobs_021226.yml"

results = run_all_jobs(str(CONFIG_PATH))


2026-02-12 10:59:21,213 [INFO] src.runner - Starting job: VZW_SOL_trust_building
2026-02-12 10:59:21,225 [INFO] pyhive.presto - SELECT 
--CAST(week_stop_date AS DATE) AS week_,
--date,
expert_id,
-- year_month,
metric,
icp_client,
--tenure_group,
site,
SUM(numerator) AS num,
SUM(denominator) AS den,
ROUND(
COALESCE(
CAST(SUM(numerator) AS DOUBLE) /
NULLIF(CAST(SUM(denominator) AS DOUBLE), 0.0),
0.000
),
3
) AS calc
FROM 
hive.care.expert_performance_metrics a
LEFT OUTER JOIN 
hive.care.l4_asurion_umt_ppx_pay_calendar d ON a."date" = CAST(d.event_date AS DATE)
WHERE 
LOWER(metric) = 'erp'
AND icp_client = 'PSS-Verizon'
AND a."date" between DATE '2026-01-30' and DATE '2026-02-13' 
 --and expert_id in ()
GROUP BY 1, 2, 3, 4


2026-02-12 10:59:32,481 [INFO] src.runner - [VZW_SOL_trust_building] Raw query rows: 2276
2026-02-12 10:59:32,488 [INFO] src.runner - [VZW_SOL_trust_building] After validity filters rows: 2070
2026-02-12 10:59:32,502 [INFO] src.runner - [VZW_SOL_trust_building] After

In [37]:
print("\n===== JOB SUMMARY =====\n")

summary_df = pd.DataFrame([
    {
        "job_name": job_name,
        "rows_raw": meta.get("rows_raw"),
        "rows_valid": meta.get("rows_valid"),
        "rows_eligible": meta.get("rows_eligible"),
        "rows_out": meta.get("rows_out"),
        "status": meta.get("status"),
        "output_file": meta.get("out_path"),
    }
    for job_name, meta in results.items()
])

summary_df



===== JOB SUMMARY =====



,job_name,rows_raw,rows_valid,rows_eligible,rows_out,status,output_file
0,VZW_SOL_trust_building,2276,2070,1862,68,success,C:\Users\nathan.taylor\Asurion\Coaching & Tech...
